In [3]:
%pip install torch
%pip install merlinquantum
%pip install perceval-quandela

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import perceval as pcvl
from perceval.components import BS, PS, Circuit
from perceval.backends import BackendFactory

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [7]:
ds = load_dataset(
    "Quandela/Challenge_Swaptions",
    data_files="level-1_Future_prediction/train.csv",
    split="train",
)

df = pd.DataFrame(ds)
df = df.drop(columns=["Date"])

data = df.values.astype(np.float64)
print("Surface shape:", data.shape)

Surface shape: (494, 224)


In [8]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

r = 5
pca = PCA(n_components=r)
factors = pca.fit_transform(data_scaled)

print("Factor shape:", factors.shape)
print("Explained variance:", np.sum(pca.explained_variance_ratio_))

Factor shape: (494, 5)
Explained variance: 0.9997859107422641


In [9]:
X = factors[:-1]
Y = factors[1:]

T = len(X)
split = int(0.8 * T)

X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

In [10]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
Y_train = torch.tensor(Y_train, dtype=torch.float32).to(device)
X_test  = torch.tensor(X_test,  dtype=torch.float32).to(device)
Y_test  = torch.tensor(Y_test,  dtype=torch.float32).to(device)

In [11]:
class ESN(nn.Module):
    def __init__(self, input_dim, reservoir_size=300, spectral_radius=0.95, sparsity=0.1):
        super().__init__()

        self.N = reservoir_size

        # Input weights
        self.Win = nn.Parameter(
            torch.randn(reservoir_size, input_dim) * 0.5,
            requires_grad=False
        )

        # Reservoir weights
        W = torch.randn(reservoir_size, reservoir_size)

        mask = (torch.rand(reservoir_size, reservoir_size) < sparsity).float()
        W *= mask

        # Spectral radius scaling
        eigvals = torch.linalg.eigvals(W)
        max_eig = torch.max(torch.abs(eigvals))
        W *= spectral_radius / max_eig.real

        self.W = nn.Parameter(W, requires_grad=False)

        # Trainable readout
        self.Wout = nn.Linear(reservoir_size, input_dim)

    def forward(self, X):
        """
        X: (T, input_dim)
        """
        T = X.shape[0]
        h = torch.zeros(self.N, device=X.device)

        outputs = []

        for t in range(T):
            h = torch.tanh(self.W @ h + self.Win @ X[t])
            y = self.Wout(h)
            outputs.append(y)

        return torch.stack(outputs)

In [12]:
model = ESN(input_dim=r, reservoir_size=400).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.Wout.parameters(), lr=1e-3)

epochs = 200

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    output = model(X_train)
    loss = criterion(output, Y_train)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

Epoch 0 | Loss: 49.646976
Epoch 20 | Loss: 22.601320
Epoch 40 | Loss: 11.225628
Epoch 60 | Loss: 8.286940
Epoch 80 | Loss: 7.608620
Epoch 100 | Loss: 7.239892
Epoch 120 | Loss: 6.937293
Epoch 140 | Loss: 6.683617
Epoch 160 | Loss: 6.463545
Epoch 180 | Loss: 6.266377


In [13]:
model.eval()
with torch.no_grad():
    Y_pred = model(X_test)
    test_loss = criterion(Y_pred, Y_test)

print("Test Factor MSE:", test_loss.item())

Test Factor MSE: 6.0478692054748535


In [14]:
Y_pred_np = Y_pred.cpu().numpy()
Y_test_np = Y_test.cpu().numpy()

# Inverse PCA
Y_pred_surface_scaled = pca.inverse_transform(Y_pred_np)
Y_test_surface_scaled = pca.inverse_transform(Y_test_np)

# Undo scaling
Y_pred_surface = scaler.inverse_transform(Y_pred_surface_scaled)
Y_test_surface = scaler.inverse_transform(Y_test_surface_scaled)

surface_mse = np.mean((Y_pred_surface - Y_test_surface)**2)

print("Surface MSE:", surface_mse)

Surface MSE: 5.0043123347865615e-05


In [15]:
def compute_qlike(forecasts, actuals):
    """
    Proper QLIKE for volatility forecasting
    """
    eps = 1e-12

    forecasts = np.maximum(np.abs(forecasts), eps)
    actuals = np.maximum(np.abs(actuals), eps)

    ratio = actuals / forecasts

    return np.mean(ratio - np.log(ratio) - 1)

In [16]:
true_var = Y_test_surface**2
pred_var = Y_pred_surface**2

qlike_value = compute_qlike(pred_var.flatten(), true_var.flatten())

print("QLIKE:", qlike_value)

QLIKE: 0.003444061113303676
